# Part 2 - Ridge, Lasso & ElasticNet Regularization
**Gray Interface '26 | Task 2**

Dataset: [Ames Housing Dataset](https://www.kaggle.com/datasets/shashanknecrothapa/ames-housing-dataset)

## Importing the dependencies

In [ ]:
!pip install kagglehub --quiet


In [ ]:
import kagglehub
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import SGDRegressor, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## Loading the Dataset

In [ ]:
path = kagglehub.dataset_download("shashanknecrothapa/ames-housing-dataset")
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

print("Shape:", df.shape)
df.head()


## Exploratory Data Analysis

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Columns with missing values:")
print(missing)


In [ ]:
# Distribution of target variable (SalePrice)
plt.figure(figsize=(8, 5))
sns.histplot(df['SalePrice'], bins=50, kde=True, color='steelblue')
plt.title('Distribution of Sale Price')
plt.xlabel('Sale Price')
plt.tight_layout()
plt.show()

print(f"Mean:   ${df['SalePrice'].mean():,.0f}")
print(f"Median: ${df['SalePrice'].median():,.0f}")


In [ ]:
# Correlation of numeric features with SalePrice
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()['SalePrice'].drop('SalePrice').sort_values(ascending=False)

plt.figure(figsize=(10, 6))
corr.head(15).plot(kind='bar', color='steelblue')
plt.title('Top 15 Features Correlated with Sale Price')
plt.ylabel('Correlation')
plt.tight_layout()
plt.show()


## Data Preprocessing

In [ ]:
# Drop columns with more than 40% missing values
threshold = 0.4 * len(df)
df = df.dropna(thresh=threshold, axis=1)

# Fill remaining numeric NaNs with column median
for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical NaNs with 'Unknown'
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna('Unknown')

print("Missing values after cleaning:", df.isnull().sum().sum())


In [ ]:
# Encode categorical columns using LabelEncoder
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

print("Shape after encoding:", df.shape)


In [ ]:
# Separate features and target
X = df.drop(columns=['SalePrice'])
y = df['SalePrice']

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features — important for SGDRegressor and regularized models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")


## Why SGDRegressor instead of LinearRegression?

`LinearRegression` computes the exact solution using the Normal Equation: **(XᵀX)⁻¹Xᵀy**.
This becomes very slow (or impossible) when the number of features is large because it requires inverting a matrix.

`SGDRegressor` uses **Stochastic Gradient Descent** — it updates weights sample by sample rather than solving the full equation at once. This scales much better to large datasets and also makes it easy to add regularization (Ridge/Lasso) just by changing one parameter.


## Training the Models

In [ ]:
def evaluate(model, X_tr, X_te, y_tr, y_te, name):
    """Train model and print MAE, MSE, RMSE, R2 for train and test"""
    model.fit(X_tr, y_tr)

    for X_, y_, label in [(X_tr, y_tr, 'Train'), (X_te, y_te, 'Test')]:
        preds = model.predict(X_)
        mae   = mean_absolute_error(y_, preds)
        mse   = mean_squared_error(y_, preds)
        rmse  = np.sqrt(mse)
        r2    = r2_score(y_, preds)
        print(f"{name} | {label} -> MAE: {mae:.1f}  MSE: {mse:.1f}  RMSE: {rmse:.1f}  R2: {r2:.4f}")
    print()
    return model


In [ ]:
# Baseline: SGDRegressor (Linear Regression via Gradient Descent)
sgd = evaluate(
    SGDRegressor(max_iter=1000, random_state=42),
    X_train_scaled, X_test_scaled, y_train, y_test,
    "SGDRegressor (baseline)"
)


In [ ]:
# Ridge Regression — penalizes large weights using L2 norm
# Use cross-validation to find the best alpha
alphas = [0.1, 1.0, 10.0, 100.0, 1000.0]
cv_scores = []

for a in alphas:
    ridge = Ridge(alpha=a)
    score = cross_val_score(ridge, X_train_scaled, y_train, cv=5, scoring='r2').mean()
    cv_scores.append(score)
    print(f"Ridge alpha={a:7.1f}  CV R2: {score:.4f}")

best_alpha_ridge = alphas[np.argmax(cv_scores)]
print(f"
Best alpha for Ridge: {best_alpha_ridge}")


In [ ]:
# Train Ridge with best alpha
ridge = evaluate(
    Ridge(alpha=best_alpha_ridge),
    X_train_scaled, X_test_scaled, y_train, y_test,
    f"Ridge (alpha={best_alpha_ridge})"
)


In [ ]:
# Lasso Regression — penalizes using L1 norm, can shrink weights to exactly 0 (feature selection)
lasso_alphas = [0.1, 1.0, 10.0, 100.0]
cv_scores_lasso = []

for a in lasso_alphas:
    lasso = Lasso(alpha=a, max_iter=5000)
    score = cross_val_score(lasso, X_train_scaled, y_train, cv=5, scoring='r2').mean()
    cv_scores_lasso.append(score)
    print(f"Lasso alpha={a:6.1f}  CV R2: {score:.4f}")

best_alpha_lasso = lasso_alphas[np.argmax(cv_scores_lasso)]
print(f"
Best alpha for Lasso: {best_alpha_lasso}")


In [ ]:
lasso = evaluate(
    Lasso(alpha=best_alpha_lasso, max_iter=5000),
    X_train_scaled, X_test_scaled, y_train, y_test,
    f"Lasso (alpha={best_alpha_lasso})"
)


In [ ]:
# ElasticNet — combines L1 and L2 penalties (best of both Ridge and Lasso)
enet = evaluate(
    ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=5000),
    X_train_scaled, X_test_scaled, y_train, y_test,
    "ElasticNet (alpha=1.0, l1_ratio=0.5)"
)


## Comparing All Models

In [ ]:
# Collect test metrics for all models into a comparison table
models = {
    'SGDRegressor': SGDRegressor(max_iter=1000, random_state=42),
    f'Ridge (a={best_alpha_ridge})': Ridge(alpha=best_alpha_ridge),
    f'Lasso (a={best_alpha_lasso})': Lasso(alpha=best_alpha_lasso, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=5000),
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    results.append({
        'Model': name,
        'MAE':  round(mean_absolute_error(y_test, preds), 1),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, preds)), 1),
        'R2':   round(r2_score(y_test, preds), 4),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


In [ ]:
# Bar chart comparison of R2 scores
plt.figure(figsize=(9, 5))
plt.bar(results_df['Model'], results_df['R2'], color=['#4C72B0','#DD8452','#55A868','#C44E52'])
plt.title('Model Comparison — R² Score on Test Set')
plt.ylabel('R²')
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## Observations

- **SGDRegressor (baseline)** gives a reasonable R² but can be unstable without regularization — its weights may be very large.
- **Ridge** shrinks all weights toward zero but keeps them all. Best when most features are useful.
- **Lasso** drives some weights to exactly zero, acting as automatic feature selection. Useful when you suspect many features are irrelevant.
- **ElasticNet** combines both — useful when there are correlated features (L1 alone can be unstable with correlated inputs).
- The best model is whichever gives the highest test R² — check the comparison table above.
- Cross-validation was used to pick regularization strength (alpha) for Ridge and Lasso to avoid overfitting to the training set.
